# Grounded Legal Document Drafting System

In [1]:
!pip install transformers pymupdf pytesseract pdf2image pillow langchain-text-splitters sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.5 MB/s eta 0:00:00


In [4]:
import fitz
import pytesseract
from PIL import Image, ImageFilter, ImageOps
import io


In [5]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(token=hf_token)

print("Successfully logged in to Hugging Face.")

Successfully logged in to Hugging Face.


In [8]:
from transformers import pipeline
import torch

pipe = pipeline("text-generation", model="google/gemma-3-4b-it", device="cpu")

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]

output = pipe(messages, max_new_tokens=50)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:
output

[[{'generated_text': [{'role': 'system',
     'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]},
    {'role': 'user',
     'content': [{'type': 'text',
       'text': 'Write a poem on Hugging Face, the company'}]},
    {'role': 'assistant',
     'content': "Okay, here's a poem about Hugging Face, aiming for a blend of admiration and a touch of technical wonder:\n\n**The Neural Bloom**\n\nWithin the cloud, a vibrant hue,\nHugging Face, a digital view.\n"}]}]]

In [9]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.9 MB/s eta 0:00:00


In [23]:
import requests
import os
import json

groq_api_key = userdata.get("GROQ_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

# Check if the request was successful
response.raise_for_status()

# Get the JSON content from the response
pretty_json = json.dumps(response.json(), indent=4)
print(pretty_json)

{
    "object": "list",
    "data": [
        {
            "id": "meta-llama/llama-4-scout-17b-16e-instruct",
            "object": "model",
            "created": 1743874824,
            "owned_by": "Meta",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 8192
        },
        {
            "id": "llama-3.1-8b-instant",
            "object": "model",
            "created": 1693721698,
            "owned_by": "Meta",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 131072
        },
        {
            "id": "canopylabs/orpheus-arabic-saudi",
            "object": "model",
            "created": 1765926439,
            "owned_by": "Canopy Labs",
            "active": true,
            "context_window": 4000,
            "public_apps": null,
            "max_completion_tokens": 50000
        },
        {
      

In [24]:
from groq import Groq

client = Groq(api_key=groq_api_key)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Write a poem on Hugging Face, the company"
        }
    ]
)

print(response.choices[0].message.content)

In Paris's heart, a spark took flight,
A company born, with a noble sight.
Hugging Face, a name that's clear,
Embracing AI, with passion and cheer.

Yann LeCun, a visionary bold,
Founded the company, where dreams unfold.
Transformers rose, a framework grand,
A revolution in AI, at its command.

Model hub, where knowledge is shared,
With a click, or a code, no need to be spared.
Fine-tuning models, to suit your need,
With ease and efficiency, as their creed.

From NLP to computer vision, they stride,
Unlocking knowledge, with AI's inside.
With datasets and notebooks, a developer's treat,
Hugging Face, where AI meets street.

Their motto is clear, to 'hug' and to share,
The power of AI, without a single care.
Open-source and inclusive, their heart's desire,
Empowering innovators, like a blazing fire.

Hugging Face, a company that shines so bright,
Illuminating AI, with a beacon light.
Innovating and pushing, the boundaries so wide,
Transforming industries, with AI's inside.

With every m

In [26]:
import fitz

doc = fitz.open("/content/sample_data/sample_legal_case_packet.pdf")

text = ""

for page in doc:
    text += page.get_text()

print(text)

Case File: Harper vs. Westbrook Holdings
This packet contains partially structured legal-style documents intended for OCR, retrieval, grounded
summarization, and evidence extraction experiments. Some sections intentionally contain inconsistent
formatting and noisy data.
Case Summary
Plaintiff: Amelia Harper
Defendant: Westbrook Holdings LLC
Case Type: Property Ownership Dispute
Relevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection
request submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer
records contain conflicting ownership information.
Document
Page
Evidence Snippet
Transfer Record
3
Ownership remains under review
Inspection Memo
5
Boundary markers unclear
Witness Statement
7
Prior owner disputed transaction
SCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture
mismatch on record copy. Need manual verificatoin before filing recommendation.”
Attached Field Note S

In [28]:
import fitz
import pytesseract
from PIL import Image, ImageOps, ImageFilter
import io

PDF_PATH = "/content/sample_data/sample_legal_case_packet.pdf"

doc = fitz.open(PDF_PATH)

all_pages = []

for page_num, page in enumerate(doc, start=1):

    # -----------------------------------
    # Native text extraction
    # -----------------------------------
    native_text = page.get_text().strip()

    # -----------------------------------
    # Find embedded images
    # -----------------------------------
    image_list = page.get_images(full=True)

    ocr_texts = []
    used_ocr_for_page = False  # Initialize flag for current page

    for img_index, img in enumerate(image_list):

        xref = img[0]

        # Extract image bytes
        base_image = doc.extract_image(xref)

        image_bytes = base_image["image"]

        image = Image.open(io.BytesIO(image_bytes))

        # -----------------------------------
        # Preprocessing
        # -----------------------------------
        image = ImageOps.grayscale(image)

        image = ImageOps.autocontrast(image)

        image = image.filter(ImageFilter.SHARPEN)

        # -----------------------------------
        # OCR image ONLY
        # -----------------------------------
        text = pytesseract.image_to_string(image)

        if text.strip():
            ocr_texts.append(text)
            used_ocr_for_page = True  # Set flag if OCR text is found

    # -----------------------------------
    # Merge native + OCR image text
    # -----------------------------------
    final_text = native_text

    if ocr_texts:

        final_text += "\n\n[OCR IMAGE CONTENT]\n"

        final_text += "\n".join(ocr_texts)

    all_pages.append({
        "page": page_num,
        "num_images": len(image_list),
        "text": final_text,
        "used_ocr": used_ocr_for_page  # Add the new key
    })

# -----------------------------------
# Preview
# -----------------------------------

for page_data in all_pages:

    print("=" * 80)
    print(f"PAGE: {page_data['page']}")
    print(f"IMAGES FOUND: {page_data['num_images']}")
    print("=" * 80)

    print(page_data["text"][:2000])
    print("\n")

PAGE: 1
IMAGES FOUND: 1
Case File: Harper vs. Westbrook Holdings
This packet contains partially structured legal-style documents intended for OCR, retrieval, grounded
summarization, and evidence extraction experiments. Some sections intentionally contain inconsistent
formatting and noisy data.
Case Summary
Plaintiff: Amelia Harper
Defendant: Westbrook Holdings LLC
Case Type: Property Ownership Dispute
Relevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection
request submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer
records contain conflicting ownership information.
Document
Page
Evidence Snippet
Transfer Record
3
Ownership remains under review
Inspection Memo
5
Boundary markers unclear
Witness Statement
7
Prior owner disputed transaction
SCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture
mismatch on record copy. Need manual verificatoin before filing recommendation

In [29]:
documents = []

for page_data in all_pages:

    documents.append({
        "text": page_data["text"],
        "metadata": {
            "page": page_data["page"],
            "source": PDF_PATH
        }
    })
documents

[{'text': 'Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Record\n3\nOwnership remains under review\nInspection Memo\n5\nBoundary markers unclear\nWitness Statement\n7\nPrior owner disputed transaction\nSCANNED NOTE (low quality transcription): “Owner ship certifcate appears incomplete. signture\nmismatch on record copy. Need manual verificatoin before filing rec

##Split Document

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", "."," "]

)


In [31]:
chunks = []

for doc in documents:

    split_texts = splitter.split_text(doc["text"])
    print(split_texts)

    for chunk in split_texts:

        chunks.append({
            "text": chunk,
            "metadata": doc["metadata"]
        })

['Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection', 'Plaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Record\n3\nOwnership remains under review\nInspection Memo\n5\nBoundary markers unclear\nWitness Statement\n7', 'rec

In [32]:
chunks

[{'text': 'Case File: Harper vs. Westbrook Holdings\nThis packet contains partially structured legal-style documents intended for OCR, retrieval, grounded\nsummarization, and evidence extraction experiments. Some sections intentionally contain inconsistent\nformatting and noisy data.\nCase Summary\nPlaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection',
  'metadata': {'page': 1,
   'source': '/content/sample_data/sample_legal_case_packet.pdf'}},
 {'text': 'Plaintiff: Amelia Harper\nDefendant: Westbrook Holdings LLC\nCase Type: Property Ownership Dispute\nRelevant Dates: March 12, 2022 — Property transfer allegedly recorded. June 4, 2023 — Inspection\nrequest submitted. January 11, 2024 — Ownership challenge filed. The claimant alleges the transfer\nrecords contain conflicting ownership information.\nDocument\nPage\nEvidence Snippet\nTransfer Rec

## Create Embeddings

In [33]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## FAISS Vector DB

In [34]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)

index.add(np.array(embeddings).astype("float32"))

In [35]:
query = "Summarize ownership dispute evidence"
query_embedding = embedding_model.encode([query])
D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)
retrieved_chunks = [chunks[i] for i in I[0]]


In [36]:
context = ""

for i, chunk in enumerate(retrieved_chunks):

    context += f"""
    [Evidence {i+1}]
    Source Page: {chunk['metadata']['page']}

    {chunk['text']}
    """

In [37]:
prompt = f"""
You are generating a grounded legal-style summary.

ONLY use the provided evidence.

If information is uncertain,
explicitly say uncertain.

Retrieved Evidence:

{context}

Generate:
1. Case summary
2. Key timeline
3. Important unresolved issues
4. Evidence references
"""

In [40]:
from groq import Groq

client = Groq(api_key=groq_api_key)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": prompt
        },
    ]
)

print(response.choices[0].message.content)

**Case Summary**

This case involves a property ownership dispute between Amelia Harper (Plaintiff) and Westbrook Holdings LLC (Defendant). The dispute revolves around alleged conflicting ownership information in the property transfer records, which were allegedly recorded on March 12, 2022. The Plaintiff challenged the ownership on January 11, 2024.

**Key Timeline**

1. March 12, 2022: Property transfer allegedly recorded
2. June 4, 2023: Inspection request submitted
3. January 11, 2024: Ownership challenge filed

**Important Unresolved Issues**

1. Conflict in ownership information in property transfer records
2. Incomplete or unclear signature on ownership certificate (as per Scanned Note)
3. Unclear boundary markers (as per Inspection Memo)
4. Potential need for manual verification of ownership certificate

**Evidence References**

1. **Evidence 1**: Transfer Record (Page 3) - "Ownership remains under review"
2. **Evidence 2**: Inspection Memo (Page 5) - "Boundary markers unclear"